# ListenBrainz Popularity Prediction

Predict which artists and tracks will reach **≥95th growth percentile** over upcoming 7-day and 30-day windows.

**Models**: XGBoost (classification + regression) and LSTM (PyTorch)  
**Data**: RDS Postgres → PySpark feature engineering → pandas → model training  
**Scope**: Separate artist and track models

| Phase | Description |
|-------|-------------|
| 1 | Data loading & feature engineering (Spark JDBC) |
| 2 | XGBoost classifier + regressor |
| 3 | LSTM sequence model (PyTorch) |
| 4 | Evaluation & comparison |

**Before running, make sure:**

- SSH tunnel is active (localhost:5433 → RDS)
- .env has fresh AWS credentials
- Required packages are installed: xgboost, shap, torch, scikit-learn, matplotlib

In [ ]:
import importlib
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Add parent dir so we can import project modules
sys.path.insert(0, str(Path.cwd().parent))

import utils
importlib.reload(utils)
from utils import load_db_credentials

print("Imports OK")

In [ ]:
from dotenv import load_dotenv

env_path = Path.cwd().parent.parent / ".env"
print(f"Loading env from: {env_path}  (exists={env_path.exists()})")
load_dotenv(dotenv_path=env_path, override=True)

# Reset boto3 session so it picks up fresh env vars
import boto3
boto3.DEFAULT_SESSION = None

creds = load_db_credentials()
print(f"DB: {creds['host']}:{creds['port']}/{creds['dbname']} as {creds['user']}")

## Phase 1 — Data Loading & Feature Engineering

Load `artist_daily_stats` / `track_daily_stats` joined with raw listen counts via Spark JDBC, engineer momentum/velocity/lag features, build forward-looking target labels, and convert to pandas with a temporal train/test split.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("ListenBrainz Popularity")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

jdbc_url = f"jdbc:postgresql://{creds['host']}:{creds['port']}/{creds['dbname']}"
jdbc_props = {
    "user": creds["user"],
    "password": creds["password"],
    "driver": "org.postgresql.Driver",
}

print(f"Spark session ready — JDBC → {creds['host']}:{creds['port']}/{creds['dbname']}")

In [ ]:
def load_and_engineer(spark, jdbc_url, jdbc_props, stats_table, listens_table, id_col):
    """
    Load stats + daily listens from Postgres, join, and engineer features.
    Returns a Spark DataFrame with all original + new columns.
    """
    stats = spark.read.jdbc(url=jdbc_url, table=stats_table, properties=jdbc_props)
    listens = spark.read.jdbc(url=jdbc_url, table=listens_table, properties=jdbc_props)
    print(f"Loaded {stats.count():,} rows from {stats_table}, {listens.count():,} from {listens_table}")

    # Join to get raw listen_count alongside precomputed stats
    df = stats.join(listens, on=["day", id_col], how="inner")

    # Windows for additional features
    w_id = Window.partitionBy(id_col).orderBy("day")
    w_id_7 = w_id.rowsBetween(-6, 0)
    w_id_30 = w_id.rowsBetween(-29, 0)
    w_first = Window.partitionBy(id_col)

    df = (
        df
        # Momentum: short-term acceleration
        .withColumn("_7d_lag7", F.lag("listen_count_past_7_days", 7).over(w_id))
        .withColumn("momentum_7d",
                     F.col("listen_count_past_7_days") - F.col("_7d_lag7"))
        .withColumn("_30d_lag30", F.lag("listen_count_past_30_days", 30).over(w_id))
        .withColumn("momentum_30d",
                     F.col("listen_count_past_30_days") - F.col("_30d_lag30"))
        # Velocity ratio: short-term vs long-term activity
        .withColumn("velocity_ratio",
                     F.when(F.col("listen_count_past_30_days") > 0,
                            F.col("listen_count_past_7_days") / F.col("listen_count_past_30_days"))
                     .otherwise(None))
        # Lifecycle proxy
        .withColumn("_first_day", F.min("day").over(w_first))
        .withColumn("days_since_first_listen",
                     F.datediff(F.col("day"), F.col("_first_day")))
        # Lag features (for LSTM sequences)
        .withColumn("listen_count_lag_1d", F.lag("listen_count", 1).over(w_id))
        .withColumn("listen_count_lag_7d", F.lag("listen_count", 7).over(w_id))
        # Day of week (0=Mon .. 6=Sun)
        .withColumn("day_of_week", (F.dayofweek("day") + 5) % 7)  # Spark dayofweek is 1=Sun
        # Drop helper columns
        .drop("_7d_lag7", "_30d_lag30", "_first_day")
    )

    print(f"Feature-engineered DataFrame: {df.count():,} rows, {len(df.columns)} columns")
    return df

print("load_and_engineer() defined")

In [ ]:
artist_df = load_and_engineer(
    spark, jdbc_url, jdbc_props,
    "artist_daily_stats", "artist_daily_listens", "artist_mbid",
)
track_df = load_and_engineer(
    spark, jdbc_url, jdbc_props,
    "track_daily_stats", "track_daily_listens", "recording_id",
)

In [ ]:
def add_targets(df, id_col, threshold=0.95):
    """
    Add forward-looking target labels:
      - is_popular_7d   (1 if max growth_percentile in next 7 days >= threshold)
      - is_popular_30d  (same for 30 days)
      - future_growth_pctl_7d  (mean growth_percentile over next 7 days)
      - future_growth_pctl_30d (mean growth_percentile over next 30 days)

    Rows where the future window extends beyond available data are dropped.
    """
    w_id = Window.partitionBy(id_col).orderBy("day")
    w_fwd_7 = w_id.rowsBetween(1, 7)
    w_fwd_30 = w_id.rowsBetween(1, 30)

    # Forward-looking aggregates
    df = (
        df
        .withColumn("_max_gp_7",  F.max("growth_percentile").over(w_fwd_7))
        .withColumn("_max_gp_30", F.max("growth_percentile").over(w_fwd_30))
        .withColumn("_mean_gp_7", F.avg("growth_percentile").over(w_fwd_7))
        .withColumn("_mean_gp_30", F.avg("growth_percentile").over(w_fwd_30))
        # Count future rows to detect incomplete windows
        .withColumn("_fwd_count_7",  F.count("growth_percentile").over(w_fwd_7))
        .withColumn("_fwd_count_30", F.count("growth_percentile").over(w_fwd_30))
    )

    # Only keep rows with complete forward windows
    df = df.filter(F.col("_fwd_count_30") >= 30)

    df = (
        df
        .withColumn("is_popular_7d",
                     F.when(F.col("_max_gp_7") >= threshold, 1).otherwise(0))
        .withColumn("is_popular_30d",
                     F.when(F.col("_max_gp_30") >= threshold, 1).otherwise(0))
        .withColumn("future_growth_pctl_7d",  F.col("_mean_gp_7"))
        .withColumn("future_growth_pctl_30d", F.col("_mean_gp_30"))
        .drop("_max_gp_7", "_max_gp_30", "_mean_gp_7", "_mean_gp_30",
              "_fwd_count_7", "_fwd_count_30")
    )

    n = df.count()
    pos_7 = df.filter(F.col("is_popular_7d") == 1).count()
    pos_30 = df.filter(F.col("is_popular_30d") == 1).count()
    print(f"After target labeling: {n:,} rows")
    print(f"  is_popular_7d:  {pos_7:,} positive ({100*pos_7/n:.1f}%)")
    print(f"  is_popular_30d: {pos_30:,} positive ({100*pos_30/n:.1f}%)")
    return df

print("add_targets() defined")

In [ ]:
artist_df = add_targets(artist_df, "artist_mbid")
track_df  = add_targets(track_df,  "recording_id")

In [ ]:
def spark_to_pandas_split(sdf, id_col, test_days=30):
    """
    Convert Spark DF to pandas, do a temporal train/test split.
    Test set = last `test_days` days of data.
    Returns (train_pdf, test_pdf, feature_cols, target_cols).
    """
    pdf = sdf.toPandas()
    pdf["day"] = pd.to_datetime(pdf["day"])
    pdf = pdf.sort_values(["day", id_col]).reset_index(drop=True)

    cutoff = pdf["day"].max() - pd.Timedelta(days=test_days)
    train = pdf[pdf["day"] <= cutoff].copy()
    test  = pdf[pdf["day"] > cutoff].copy()

    # Feature columns: everything except id, day, and targets
    target_cols = ["is_popular_7d", "is_popular_30d",
                   "future_growth_pctl_7d", "future_growth_pctl_30d"]
    exclude = {id_col, "day"} | set(target_cols)
    feature_cols = [c for c in pdf.columns if c not in exclude and pdf[c].dtype in ("float64", "float32", "int64", "int32")]

    print(f"Train: {len(train):,} rows ({train['day'].min().date()} – {train['day'].max().date()})")
    print(f"Test:  {len(test):,} rows  ({test['day'].min().date()} – {test['day'].max().date()})")
    print(f"Features ({len(feature_cols)}): {feature_cols}")
    print(f"Targets: {target_cols}")
    return train, test, feature_cols, target_cols

print("spark_to_pandas_split() defined")

In [ ]:
art_train, art_test, art_features, target_cols = spark_to_pandas_split(artist_df, "artist_mbid")
trk_train, trk_test, trk_features, _          = spark_to_pandas_split(track_df,  "recording_id")

# Stop Spark — not needed for model training
spark.stop()
print("Spark session stopped.")

In [ ]:
# Quick sanity check: NaN rates and class balance
for name, df in [("Artist Train", art_train), ("Track Train", trk_train)]:
    print(f"\n{'='*40}")
    print(f"{name}  ({len(df):,} rows)")
    print(f"  NaN %: {(df[art_features if 'Artist' in name else trk_features].isna().mean() * 100).round(1).to_dict()}")
    for t in target_cols[:2]:  # classification targets
        pos = df[t].mean()
        print(f"  {t}: {100*pos:.1f}% positive")

## Phase 2 — XGBoost Models

Train XGBoost classifiers (binary: popular yes/no) and regressors (continuous growth percentile) for both 7-day and 30-day horizons. Temporal expanding-window CV for hyperparameter selection. SHAP for feature importance.

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    mean_squared_error, mean_absolute_error,
    classification_report, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import TimeSeriesSplit
import shap

print("XGBoost + sklearn + SHAP imports OK")

In [ ]:
def train_xgb_classifier(train_df, test_df, feature_cols, target_col, n_splits=3):
    """
    Train XGBClassifier with temporal CV, evaluate on held-out test set.
    Returns (model, metrics_dict, test_proba).
    """
    X_train = train_df[feature_cols].fillna(0).values
    y_train = train_df[target_col].values
    X_test  = test_df[feature_cols].fillna(0).values
    y_test  = test_df[target_col].values

    # Class imbalance weight
    pos_rate = y_train.mean()
    spw = (1 - pos_rate) / max(pos_rate, 1e-6)

    # Temporal CV for early stopping round estimation
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_rounds = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
        model = XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05,
            scale_pos_weight=spw, eval_metric="auc",
            early_stopping_rounds=30, random_state=42,
            tree_method="hist",
        )
        model.fit(
            X_train[tr_idx], y_train[tr_idx],
            eval_set=[(X_train[val_idx], y_train[val_idx])],
            verbose=False,
        )
        best_rounds.append(model.best_iteration)
        val_proba = model.predict_proba(X_train[val_idx])[:, 1]
        val_auc = roc_auc_score(y_train[val_idx], val_proba) if y_train[val_idx].sum() > 0 else float("nan")
        print(f"  Fold {fold+1}: best_iter={model.best_iteration}, val_AUC={val_auc:.4f}")

    # Final model with median best rounds
    final_n = int(np.median(best_rounds))
    model = XGBClassifier(
        n_estimators=final_n, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, eval_metric="auc",
        random_state=42, tree_method="hist",
    )
    model.fit(X_train, y_train, verbose=False)

    # Test evaluation
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= 0.5).astype(int)

    metrics = {
        "roc_auc": roc_auc_score(y_test, test_proba) if y_test.sum() > 0 else float("nan"),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "n_estimators": final_n,
    }
    print(f"  Test: AUC={metrics['roc_auc']:.4f}  F1={metrics['f1']:.4f}  "
          f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}")
    return model, metrics, test_proba


def train_xgb_regressor(train_df, test_df, feature_cols, target_col, clf_threshold=0.95, n_splits=3):
    """
    Train XGBRegressor with temporal CV, evaluate on held-out test set.
    Also computes classification metrics by thresholding predictions at clf_threshold.
    Returns (model, metrics_dict, test_preds).
    """
    X_train = train_df[feature_cols].fillna(0).values
    y_train = train_df[target_col].values
    X_test  = test_df[feature_cols].fillna(0).values
    y_test  = test_df[target_col].values

    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_rounds = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
        model = XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.05,
            eval_metric="rmse", early_stopping_rounds=30,
            random_state=42, tree_method="hist",
        )
        model.fit(
            X_train[tr_idx], y_train[tr_idx],
            eval_set=[(X_train[val_idx], y_train[val_idx])],
            verbose=False,
        )
        best_rounds.append(model.best_iteration)
        val_pred = model.predict(X_train[val_idx])
        val_rmse = np.sqrt(mean_squared_error(y_train[val_idx], val_pred))
        print(f"  Fold {fold+1}: best_iter={model.best_iteration}, val_RMSE={val_rmse:.4f}")

    final_n = int(np.median(best_rounds))
    model = XGBRegressor(
        n_estimators=final_n, max_depth=6, learning_rate=0.05,
        eval_metric="rmse", random_state=42, tree_method="hist",
    )
    model.fit(X_train, y_train, verbose=False)

    test_preds = model.predict(X_test)
    test_binary = (test_preds >= clf_threshold).astype(int)
    y_test_binary = (y_test >= clf_threshold).astype(int)

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_test, test_preds)),
        "mae": mean_absolute_error(y_test, test_preds),
        "roc_auc_thresholded": roc_auc_score(y_test_binary, test_preds) if y_test_binary.sum() > 0 else float("nan"),
        "f1_thresholded": f1_score(y_test_binary, test_binary, zero_division=0),
        "n_estimators": final_n,
    }
    print(f"  Test: RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  "
          f"threshold-AUC={metrics['roc_auc_thresholded']:.4f}  threshold-F1={metrics['f1_thresholded']:.4f}")
    return model, metrics, test_preds

print("XGBoost training functions defined")

### Train all XGBoost models
Artist and Track × 7d and 30d horizons × Classifier and Regressor = **8 models**

In [ ]:
xgb_results = {}  # key: (entity, horizon, model_type) → (model, metrics, preds)

configs = [
    ("artist", art_train, art_test, art_features),
    ("track",  trk_train, trk_test, trk_features),
]

for entity, train_df, test_df, feat_cols in configs:
    for horizon in ["7d", "30d"]:
        clf_target = f"is_popular_{horizon}"
        reg_target = f"future_growth_pctl_{horizon}"

        print(f"\n{'='*60}")
        print(f"XGB Classifier — {entity} — {horizon}")
        print(f"{'='*60}")
        m, met, proba = train_xgb_classifier(train_df, test_df, feat_cols, clf_target)
        xgb_results[(entity, horizon, "clf")] = (m, met, proba)

        print(f"\nXGB Regressor — {entity} — {horizon}")
        print(f"{'-'*60}")
        m, met, preds = train_xgb_regressor(train_df, test_df, feat_cols, reg_target)
        xgb_results[(entity, horizon, "reg")] = (m, met, preds)

print(f"\nTrained {len(xgb_results)} XGBoost models.")

### SHAP Feature Importance (XGBoost Classifiers)

In [ ]:
for entity, train_df, _, feat_cols in configs:
    for horizon in ["7d", "30d"]:
        model = xgb_results[(entity, horizon, "clf")][0]
        X_sample = train_df[feat_cols].fillna(0).sample(n=min(500, len(train_df)), random_state=42)
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_sample)

        print(f"\n{entity.title()} — {horizon} Classifier")
        shap.summary_plot(shap_vals, X_sample, plot_type="bar", max_display=15, show=True)

## Phase 3 — LSTM Sequence Models (PyTorch)

Build sliding-window sequences (T=30 past days → predict label) for each entity. Train LSTM networks for both classification and regression targets.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {device}")

In [ ]:
SEQ_LEN = 30  # Past days used as input sequence

class TimeSeriesDataset(Dataset):
    """Sliding-window dataset: (SEQ_LEN, n_features) → target scalar."""

    def __init__(self, df, id_col, feature_cols, target_col, seq_len=SEQ_LEN):
        self.sequences = []
        self.targets = []

        for _, group in df.groupby(id_col):
            group = group.sort_values("day")
            X = group[feature_cols].fillna(0).values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)

            for i in range(seq_len, len(group)):
                self.sequences.append(X[i - seq_len : i])
                self.targets.append(y[i])

        self.sequences = np.array(self.sequences)  # (N, T, F)
        self.targets = np.array(self.targets)       # (N,)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx], dtype=torch.float32),
            torch.tensor(self.targets[idx], dtype=torch.float32),
        )


class PopularityLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2, task="clf"):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden_size, 1)
        self.task = task

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        _, (h_n, _) = self.lstm(x)
        out = self.head(h_n[-1])  # last layer's hidden state
        return out.squeeze(-1)

print(f"TimeSeriesDataset and PopularityLSTM defined (SEQ_LEN={SEQ_LEN})")

In [ ]:
def train_lstm(train_df, test_df, id_col, feature_cols, target_col,
               task="clf", epochs=30, batch_size=256, lr=1e-3):
    """
    Train an LSTM model. task='clf' uses BCEWithLogitsLoss, task='reg' uses MSELoss.
    Returns (model, metrics_dict, test_preds).
    """
    train_ds = TimeSeriesDataset(train_df, id_col, feature_cols, target_col)
    test_ds  = TimeSeriesDataset(test_df,  id_col, feature_cols, target_col)
    print(f"  Train sequences: {len(train_ds):,}  Test sequences: {len(test_ds):,}")

    if len(train_ds) == 0 or len(test_ds) == 0:
        print("  ⚠ Not enough sequences — skipping")
        return None, {}, np.array([])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    n_features = train_ds.sequences.shape[2]
    model = PopularityLSTM(n_features, task=task).to(device)

    if task == "clf":
        # Handle class imbalance with pos_weight
        pos_rate = train_ds.targets.mean()
        pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(y_batch)
        train_loss /= len(train_ds)

        # Validation on test set (for early stopping only)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch)
                val_loss += criterion(out, y_batch).item() * len(y_batch)
        val_loss /= len(test_ds)
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

        if patience_counter >= 10:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    # Restore best model and evaluate
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            out = model(X_batch)
            if task == "clf":
                out = torch.sigmoid(out)
            all_preds.append(out.cpu().numpy())
            all_targets.append(y_batch.numpy())

    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)

    metrics = {}
    if task == "clf":
        binary_preds = (preds >= 0.5).astype(int)
        metrics["roc_auc"] = roc_auc_score(targets, preds) if targets.sum() > 0 else float("nan")
        metrics["f1"] = f1_score(targets, binary_preds, zero_division=0)
        metrics["precision"] = precision_score(targets, binary_preds, zero_division=0)
        metrics["recall"] = recall_score(targets, binary_preds, zero_division=0)
        print(f"  Test: AUC={metrics['roc_auc']:.4f}  F1={metrics['f1']:.4f}  "
              f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}")
    else:
        binary_preds = (preds >= 0.95).astype(int)
        binary_targets = (targets >= 0.95).astype(int)
        metrics["rmse"] = np.sqrt(mean_squared_error(targets, preds))
        metrics["mae"] = mean_absolute_error(targets, preds)
        metrics["roc_auc_thresholded"] = roc_auc_score(binary_targets, preds) if binary_targets.sum() > 0 else float("nan")
        metrics["f1_thresholded"] = f1_score(binary_targets, binary_preds, zero_division=0)
        print(f"  Test: RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  "
              f"threshold-AUC={metrics['roc_auc_thresholded']:.4f}  threshold-F1={metrics['f1_thresholded']:.4f}")

    return model, metrics, preds

print("train_lstm() defined")

### Train all LSTM models
Same structure as XGBoost: Artist/Track × 7d/30d × Classifier/Regressor

In [ ]:
lstm_results = {}  # key: (entity, horizon, model_type) → (model, metrics, preds)

lstm_configs = [
    ("artist", art_train, art_test, "artist_mbid", art_features),
    ("track",  trk_train, trk_test, "recording_id", trk_features),
]

for entity, train_df, test_df, id_col, feat_cols in lstm_configs:
    for horizon in ["7d", "30d"]:
        clf_target = f"is_popular_{horizon}"
        reg_target = f"future_growth_pctl_{horizon}"

        print(f"\n{'='*60}")
        print(f"LSTM Classifier — {entity} — {horizon}")
        print(f"{'='*60}")
        m, met, preds = train_lstm(train_df, test_df, id_col, feat_cols, clf_target, task="clf")
        lstm_results[(entity, horizon, "clf")] = (m, met, preds)

        print(f"\nLSTM Regressor — {entity} — {horizon}")
        print(f"{'-'*60}")
        m, met, preds = train_lstm(train_df, test_df, id_col, feat_cols, reg_target, task="reg")
        lstm_results[(entity, horizon, "reg")] = (m, met, preds)

print(f"\nTrained {len(lstm_results)} LSTM models.")

## Phase 4 — Evaluation & Comparison

Side-by-side metrics table, ROC curves, precision-recall curves, and top-N predicted rising entities vs actuals.

In [ ]:
# Build comparison metrics table
rows = []
for key_set, label in [(xgb_results, "XGBoost"), (lstm_results, "LSTM")]:
    for (entity, horizon, mtype), (_, metrics, _) in key_set.items():
        if not metrics:
            continue
        row = {"Model": label, "Entity": entity, "Horizon": horizon, "Type": mtype}
        row.update(metrics)
        rows.append(row)

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

### ROC & Precision-Recall Curves (Classifiers only)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10))

for col_offset, (entity, _, test_df, feat_cols) in enumerate(configs):
    for row, horizon in enumerate(["7d", "30d"]):
        target_col = f"is_popular_{horizon}"

        # --- ROC ---
        ax_roc = axes[row][col_offset * 2]
        ax_roc.set_title(f"ROC — {entity.title()} {horizon}")
        ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.3)

        # XGBoost
        xgb_proba = xgb_results[(entity, horizon, "clf")][2]
        # LSTM uses different test set length (due to sequences), need to align targets
        xgb_y = test_df[target_col].values
        fpr, tpr, _ = roc_curve(xgb_y, xgb_proba)
        ax_roc.plot(fpr, tpr, label=f"XGB (AUC={roc_auc_score(xgb_y, xgb_proba):.3f})")

        lstm_preds = lstm_results[(entity, horizon, "clf")][2]
        if len(lstm_preds) > 0:
            # LSTM test set is smaller (seq_len trimming), build aligned targets
            lstm_ds = TimeSeriesDataset(test_df, configs[col_offset][0] == "artist" and "artist_mbid" or "recording_id",
                                        feat_cols, target_col)
            lstm_y = lstm_ds.targets
            fpr2, tpr2, _ = roc_curve(lstm_y, lstm_preds)
            ax_roc.plot(fpr2, tpr2, label=f"LSTM (AUC={roc_auc_score(lstm_y, lstm_preds):.3f})")

        ax_roc.legend(fontsize=9)
        ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")

        # --- Precision-Recall ---
        ax_pr = axes[row][col_offset * 2 + 1]
        ax_pr.set_title(f"PR — {entity.title()} {horizon}")

        prec, rec, _ = precision_recall_curve(xgb_y, xgb_proba)
        ax_pr.plot(rec, prec, label="XGB")

        if len(lstm_preds) > 0:
            prec2, rec2, _ = precision_recall_curve(lstm_y, lstm_preds)
            ax_pr.plot(rec2, prec2, label="LSTM")

        ax_pr.legend(fontsize=9)
        ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")

plt.tight_layout()
plt.show()

### Top-N Predicted Rising Entities vs Actual
Show the top 20 entities predicted most likely to be "popular" and whether they actually were.

In [ ]:
TOP_N = 20

for entity, _, test_df, feat_cols in configs:
    id_col = "artist_mbid" if entity == "artist" else "recording_id"
    for horizon in ["7d", "30d"]:
        target_col = f"is_popular_{horizon}"
        proba = xgb_results[(entity, horizon, "clf")][2]

        result = test_df[[id_col, "day", target_col]].copy()
        result["xgb_score"] = proba
        # Aggregate: mean predicted score per entity
        entity_scores = (
            result.groupby(id_col)
            .agg(
                mean_xgb_score=("xgb_score", "mean"),
                actually_popular=(target_col, "max"),
                n_days=("day", "count"),
            )
            .sort_values("mean_xgb_score", ascending=False)
            .head(TOP_N)
        )
        hit_rate = entity_scores["actually_popular"].mean()
        print(f"\n{entity.title()} — {horizon}: Top {TOP_N} predicted (hit rate: {100*hit_rate:.0f}%)")
        display(entity_scores.round(4))